# Семинар 01. Базовые алгоритмы и оценка сложности

## Цели

После семинара вы сможете:

- определить вход, результат и граничные случаи задачи;
- сравнить два решения по числу операций и дополнительной памяти;
- найти скрытый обход списка во встроенной операции;
- обосновать корректность и сложность своего алгоритма.

## Перед началом

Нужны переменные, условия, циклы, функции и списки. Аннотации вроде
`list[int]` описывают ожидаемые типы, но сами не проверяют входные данные.

**Основная часть — ориентировочно 90 минут:** контракт задачи — 5;
максимум и сложность — 20; наивный поиск и счётчик операций — 15;
задание 1 — 25; битовые операции — 10; задание 2 — 10; итоги — 5.
Бонусные задания и справка находятся после основной части.

Сначала отвечайте на вопросы к примеру, затем запускайте код.
В заготовках нужно написать реализацию. Проверки определены отдельными
функциями: после решения раскомментируйте вызов в конце ячейки проверки.
До этого последовательный запуск ноутбука выполняет демонстрации и создаёт
заготовки, но не проверяет нерешённые задачи.

## Программа, алгоритм и контракт

Алгоритм описывает способ решения задачи, а программа переводит этот способ
в команды для конкретной среды. Рецепт один, но на каждой кухне приходится
учитывать свою плиту и посуду. К алгоритму в программе добавляются ввод,
вывод и взаимодействие с другими частями системы.

До реализации договоритесь о четырёх вещах: допустимых входных данных,
результате, граничных случаях и условии завершения. «Найти максимум»
недостаточно: нужно хотя бы решить, что делать с пустым списком.

**Обсудите:** чем отличаются условия «список целых чисел» и «непустой список
положительных целых чисел»? Как это влияет на реализацию?

## Пример: поиск максимального элемента

**Контракт:** на входе список целых чисел; результат — наибольшее значение,
для пустого списка — `None`. Вход не изменяем.

В непустом списке запоминаем первый элемент как текущий максимум,
затем сравниваем с ним каждый следующий и при необходимости заменяем.

**До запуска:** найдите результаты для `[]`, `[7]`, `[-8, -3, -11]`
и `[4, 4, 2]`. Что изменится, если начать с `current_max = 0`?

In [ ]:
def find_max(items: list[int]) -> int | None:
    if not items:
        return None
    current_max = items[0]
    for index in range(1, len(items)):
        if items[index] > current_max:
            current_max = items[index]
    return current_max


for items in [[], [7], [-8, -3, -11], [4, 4, 2]]:
    print(items, '->', find_max(items))

assert find_max([]) is None
assert find_max([7]) == 7
assert find_max([-8, -3, -11]) == -3
assert find_max([4, 4, 2]) == 4


После обработки очередного элемента `current_max` равен максимуму уже
просмотренной части. Это утверждение верно в начале, сохраняется на каждом
шаге и после последнего шага даёт нужный результат. Такое сохраняющееся
утверждение называют **инвариантом**.

Начальное значение `0` нарушает это рассуждение: нуля может не быть во входе,
и для списка отрицательных чисел получится неверный ответ.

**Измените пример:** добавьте случай, когда максимум находится в начале,
и случай, когда он находится в конце. Можно ли завершить поиск раньше?

In [ ]:
# В прикладном коде используйте встроенную функцию.
assert max([-8, -3, -11]) == -3
assert max([], default=None) is None

# Намеренно вызываем и перехватываем ошибку: default не указан.
try:
    max([])
except ValueError as error:
    print(type(error).__name__, error)


## Время и дополнительная память

Размер входа здесь — `N = len(items)`. Временная сложность описывает рост
числа операций при увеличении `N`, а пространственная — рост расхода памяти.
Дополнительную память считаем без исходного списка.

Для непустого списка поиск максимума делает ровно `N - 1` сравнений.
Несколько переменных занимают постоянное число ячеек памяти независимо от `N`.
Поэтому время — `Θ(N)`, дополнительная память — `Θ(1)`.

`O(f(N))` задаёт асимптотическую верхнюю границу: при достаточно больших `N`
затраты не превышают некоторую константу, умноженную на `f(N)`.
`Θ(f(N))` означает точный порядок роста. Линейный алгоритм принадлежит
и `O(N²)`, но такая граница слишком груба для сравнения решений.
Постоянные множители и младшие слагаемые не меняют порядок роста:
`3N + 10 = Θ(N)`, `N² + N = Θ(N²)`.

Один и тот же порядок роста не означает одинаковое время в секундах.
Кроме того, нужно указать, какой случай рассматриваем: лучший или худший.
Для максимума всё равно нужны все `N - 1` сравнений, а поиск заданного
значения может остановиться сразу или дойти до конца.

В задачах семинара считаем одну арифметическую или битовую операцию над
числом элементарной, а память — в числовых ячейках. Это учебная модель;
стоимость работы с очень длинными целыми в Python отдельно обсуждается в справке.

**До запуска:** сколько сравнений будет для 4 и 8 элементов?

In [ ]:
def find_max_counted(items: list[int]) -> tuple[int | None, int]:
    if not items:
        return None, 0
    current_max = items[0]
    comparisons = 0
    for index in range(1, len(items)):
        comparisons += 1
        if items[index] > current_max:
            current_max = items[index]
    return current_max, comparisons


for n in [1, 4, 8, 16]:
    maximum, comparisons = find_max_counted(list(range(n)))
    assert comparisons == n - 1
    print(f'N={n:2}: максимум={maximum:2}, сравнений={comparisons:2}')


## Самопроверка

1. Почему нельзя пропустить последний элемент при поиске максимума?
2. Как изменится дополнительная память, если заменить обход по индексам
   на `for item in items[1:]`?
3. Сколько стоит получить максимум из уже отсортированного списка?
   А если список ещё нужно отсортировать?
4. Почему число сравнений не равно общему числу инструкций программы?

Срез списка создаёт новый список со ссылками на элементы: у `items[1:]`
линейные время и дополнительная память. Доступ к последнему элементу
готового отсортированного списка постоянный, но предварительная сортировка
добавляет работу. Ради одного максимума достаточно одного прохода.

## Разбор: одно пропущенное число

Дан список из `N` различных целых чисел от `1` до `N + 1`, ровно одно
отсутствует. Порядок произвольный. Например, для `[4, 1, 2]` ответ — `3`,
а для пустого списка — `1`.

**До запуска:** проследите выполнение следующей функции для `[4, 1, 2]`.
Сколько работы скрывается за одной проверкой `candidate not in items`?

In [ ]:
def find_missing_number_naive(items: list[int]) -> int:
    for candidate in range(1, len(items) + 2):
        if candidate not in items:
            return candidate
    raise ValueError('Вход не соответствует условию')


assert find_missing_number_naive([4, 1, 2]) == 3
assert find_missing_number_naive([1, 2, 3]) == 4
assert find_missing_number_naive([]) == 1


## Эксперимент: считаем сравнения

Раскроем поиск в списке в явный цикл. Счётчик увеличивается только при
сравнении кандидата с элементом, а не при проверках границ циклов.
Это модель работы `in` для наших целых чисел, а не подсчёт инструкций CPython.

**До запуска:** в эксперименте отсутствует последнее допустимое число.
Как изменятся счётчики при удвоении `N`? Сначала запишите прогноз.

In [ ]:
def find_missing_number_naive_counted(items: list[int]) -> tuple[int, int]:
    comparisons = 0
    for candidate in range(1, len(items) + 2):
        found = False
        for item in items:
            comparisons += 1
            if item == candidate:
                found = True
                break
        if not found:
            return candidate, comparisons
    raise ValueError('Вход не соответствует условию')


print('   N | максимум: сравнения | пропуск: сравнения')
for n in [8, 16, 32, 64, 128]:
    items = list(range(1, n + 1))
    _, max_comparisons = find_max_counted(items)
    missing, missing_comparisons = find_missing_number_naive_counted(items)
    assert missing == n + 1
    assert missing_comparisons == n * (n + 1) // 2 + n
    print(f'{n:4} | {max_comparisons:19} | {missing_comparisons:18}')


Для этого входа число сравнений при поиске пропуска равно
`1 + 2 + ... + N + N`. Последнее слагаемое — безуспешный поиск `N + 1`.
Получаем `N(N + 1) / 2 + N`, то есть `Θ(N²)`.
У поиска максимума счётчик растёт линейно. Для обоих алгоритмов
дополнительная память постоянна; вход для эксперимента готовится отдельно.

**Проверьте себя:** замените вход эксперимента на `list(range(2, n + 2))`.
Какой случай теперь измеряется? Почему один запуск не доказывает сложность
для всех допустимых входов? При изменении входа пересмотрите и утверждения
`assert`, описывающие прежний эксперимент.

## Задание 1. Линейный поиск пропущенного числа

Дан список из `N` различных целых чисел от `1` до `N + 1` включительно.
Ровно одно число отсутствует. Верните его. Порядок элементов произвольный;
вход гарантированно соответствует условию.

**Примеры:** `[4, 1, 2] → 3`, `[3, 2, 1] → 4`, `[] → 1`.

**Требования:** время `O(N)`, дополнительная память `O(1)`, вход не меняется.
Объясните, почему алгоритм возвращает правильный ответ, и обоснуйте обе
оценки сложности. Сравните решение с `find_missing_number_naive`.

In [ ]:
def find_missing_number(items: list[int]) -> int:
    raise NotImplementedError


In [ ]:
def check_find_missing_number():
    cases = [([], 1), ([1], 2), ([2], 1), ([4, 1, 2], 3),
             ([3, 2, 1], 4), ([5, 2, 4, 3], 1)]
    for items, expected in cases:
        before = items.copy()
        assert find_missing_number(items) == expected, before
        assert items == before, 'Вход изменён'
    print('Проверки пройдены; сложность нужно обосновать отдельно.')


# check_find_missing_number()


## Битовые операции и XOR

Битовые операции применяются к соответствующим разрядам двоичной записи.
XOR даёт единицу там, где биты различаются; в Python это `^`.
Возведение в степень записывается иначе: `**`.

| Операция | Запись | Для `a = 0b101`, `b = 0b011` |
|---|---|---|
| AND | `a & b` | `0b001` |
| OR | `a \| b` | `0b111` |
| XOR | `a ^ b` | `0b110` |

Свойства XOR: `x ^ x == 0`, `x ^ 0 == x`.
Перестановка операндов и расстановка скобок не меняют результат:
`a ^ b == b ^ a`, `(a ^ b) ^ c == a ^ (b ^ c)`.

**До запуска:** вычислите `5 ^ 3` по разрядам и `5 ^ 5`.
После запуска измените `a` и `b`, затем проверьте расчёт вручную.

In [ ]:
a, b = 5, 3
print(f'{a:03b} ^ {b:03b} = {a ^ b:03b}')
for x in [0, 1, 5, -7]:
    assert x ^ x == 0
    assert x ^ 0 == x
assert (5 ^ 3) ^ 9 == 5 ^ (3 ^ 9)


## Задание 2. Элемент без пары

Дан непустой список из `2N + 1` целых чисел. Одно значение встречается
ровно один раз, остальные — ровно два. Верните значение без пары.
Допускаются ноль и отрицательные числа. Вход гарантированно корректен.

**Примеры:** `[4, 1, 2, 1, 2] → 4`, `[0] → 0`, `[-5, 2, 2] → -5`.

**Требования:** время `O(N)`, дополнительная память `O(1)`, вход не меняется.
Объясните корректность решения и роль ограничений на число повторений.

In [ ]:
def find_unique(items: list[int]) -> int:
    raise NotImplementedError


In [ ]:
def check_find_unique():
    cases = [([4, 1, 2, 1, 2], 4), ([0], 0), ([-5, 2, 2], -5),
             ([3, 0, 3], 0), ([-2, 7, -2], 7)]
    for items, expected in cases:
        before = items.copy()
        assert find_unique(items) == expected, before
        assert items == before, 'Вход изменён'
    print('Проверки пройдены; сложность нужно обосновать отдельно.')


# check_find_unique()


## Итоги

- Корректность начинается с контракта: допустимого входа, результата
  и граничных случаев.
- Для оценки сложности нужно назвать размер входа, считаемые операции
  и рассматриваемый случай.
- Один цикл в тексте программы может скрывать повторные обходы внутри `in`.
- Дополнительная память включает срезы и временные коллекции.
- Проверки на примерах находят ошибки, но не заменяют обоснование
  корректности и сложности.

**В конце занятия:** выберите одно своё решение и объясните его соседу
без запуска кода: что хранит каждая переменная, почему получается ответ
и сколько потребуется времени и памяти при росте входа.

## Дополнительная практика

Следующие три задания — усложнения после основной части.
Для каждого действуют требования `O(N)` по времени и `O(1)` по дополнительной
памяти в принятой учебной модели. Вход не меняется и гарантированно
соответствует условию. Результат — кортеж указанной длины,
порядок найденных чисел произвольный. Обоснуйте корректность и сложность.

## Бонусное задание 1. Два пропущенных числа

Дан список из `N` различных целых чисел от `1` до `N + 2` включительно.
Ровно два числа отсутствуют. Найдите их.

**Примеры:** `[] → (1, 2)`, `[1, 3] → (2, 4)`, `[4, 2, 3] → (1, 5)`.

In [ ]:
def find_two_missing_numbers(items: list[int]) -> tuple[int, int]:
    raise NotImplementedError


In [ ]:
def check_find_two_missing_numbers():
    for items, expected in [([], {1, 2}), ([1, 3], {2, 4}),
                            ([4, 2, 3], {1, 5}), ([5, 1, 3], {2, 4})]:
        before = items.copy()
        result = find_two_missing_numbers(items)
        assert isinstance(result, tuple) and len(result) == 2
        assert set(result) == expected, before
        assert items == before, 'Вход изменён'
    print('Проверки пройдены; сложность нужно обосновать отдельно.')


# check_find_two_missing_numbers()


## Бонусное задание 2. Два элемента без пары

Дан список из `2N + 2` целых чисел. Ровно два различных значения встречаются
по одному разу, остальные — ровно по два. Верните оба значения.
Допускаются ноль и отрицательные числа.

**Примеры:** `[1, 2] → (1, 2)`, `[4, 1, 2, 1, 2, 7] → (4, 7)`.

In [ ]:
def find_two_unique(items: list[int]) -> tuple[int, int]:
    raise NotImplementedError


In [ ]:
def check_find_two_unique():
    for items, expected in [([1, 2], {1, 2}),
                            ([4, 1, 2, 1, 2, 7], {4, 7}),
                            ([0, -5, 9, 9], {0, -5}),
                            ([-2, -7, 3, 3], {-2, -7})]:
        before = items.copy()
        result = find_two_unique(items)
        assert isinstance(result, tuple) and len(result) == 2
        assert set(result) == expected, before
        assert items == before, 'Вход изменён'
    print('Проверки пройдены; сложность нужно обосновать отдельно.')


# check_find_two_unique()


## Бонусное задание 3. Три элемента без пары

Дан список из `2N + 3` целых чисел. Ровно три различных значения встречаются
по одному разу, остальные — ровно по два. Верните три значения.
Допускаются ноль и отрицательные числа.

**Примеры:** `[1, 2, 3] → (1, 2, 3)`,
`[4, 1, 2, 1, 2, 7, 9] → (4, 7, 9)`.

In [ ]:
def find_three_unique(items: list[int]) -> tuple[int, int, int]:
    raise NotImplementedError


In [ ]:
def check_find_three_unique():
    for items, expected in [([1, 2, 3], {1, 2, 3}),
                            ([4, 1, 2, 1, 2, 7, 9], {4, 7, 9}),
                            ([0, -5, 6, 8, 8], {0, -5, 6}),
                            ([-1, -2, -3, 4, 4], {-1, -2, -3})]:
        before = items.copy()
        result = find_three_unique(items)
        assert isinstance(result, tuple) and len(result) == 3
        assert set(result) == expected, before
        assert items == before, 'Вход изменён'
    print('Проверки пройдены; сложность нужно обосновать отдельно.')


# check_find_three_unique()


## Справочно: границы оценок и модель вычислений

| Обозначение | Смысл |
|---|---|
| `O(f(N))` | асимптотическая верхняя граница |
| `Ω(f(N))` | асимптотическая нижняя граница |
| `Θ(f(N))` | совпадение верхней и нижней границ по порядку роста |

Эти обозначения сами по себе не выбирают лучший или худший случай:
границу можно задавать для любой из соответствующих функций затрат.
Для среднего случая нужно ещё определить вероятностное распределение входов.
Результат замера на одном списке такого распределения не задаёт.

В Python целые числа имеют произвольную точность. Если учитывать число битов,
стоимость арифметики, битовых операций и хранения одного числа зависит
от его длины. Поэтому постоянное число целочисленных переменных не означает
постоянное число байтов для неограниченно больших значений.

«Компилируемый» и «интерпретируемый» характеризуют реализацию, а не язык
сам по себе. CPython сначала компилирует исходный код в байткод, затем
выполняет его. Эта деталь не меняет способ подсчёта операций в нашем разборе.